---
## Section 1 — Imports & Configuration

All tunable parameters are consolidated here. Adjust `TRAIN_START_YEAR`,
`TRAIN_END_YEAR`, `VAL_START_YEAR`, `VAL_END_YEAR`, and `FORECAST_END_YEAR` to
shift the time windows without touching any other cell.


In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.window import Window

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder

import pandas as pd
import numpy as np

# ---------------------------------------------------------------------------
# INPUT TABLES
# ---------------------------------------------------------------------------
CLEAN_TABLE   = "fact_enrollment_annualized_clean"   # cleaned enrollment facts
DIM_TABLE     = "dim_school"                          # school metadata

# OUTPUT TABLES
FEATURE_TABLE = "ml_enrollment_feature_table_v4"     # written by Stage 1
FORECAST_TABLE_SUFFIX = "v3_fixed"                   # forecast table suffix

# ---------------------------------------------------------------------------
# YEAR WINDOWS  — edit these to move the train/val/forecast windows
# ---------------------------------------------------------------------------
TRAIN_START_YEAR    = 2020
TRAIN_END_YEAR      = 2025
VAL_START_YEAR      = 2026
VAL_END_YEAR        = 2026
FORECAST_END_YEAR   = 2027

# ---------------------------------------------------------------------------
# MODEL HYPERPARAMETERS  (fixed best-params from prior grid search)
# ---------------------------------------------------------------------------
BEST_MAX_DEPTH        = 5
BEST_MAX_ITER         = 100
BEST_STEP_SIZE        = 0.1
BEST_SUBSAMPLING_RATE = 0.8

# ---------------------------------------------------------------------------
# MISC
# ---------------------------------------------------------------------------
LABEL_COL  = "ENROLLMENT"
TIME_COL   = "SCHOOL_YEAR"
WEIGHT_COL = "sample_weight"

TARGET_GRADES = ["1", "2", "3", "4", "5", "6", "7", "8", "10", "11", "12"]
# K and Grade 9 are entry grades and need separate models (birth data / GoCPS application data).

GRADE_ORDER = {
    "PE": 0, "PK": 1, "K": 2,
    "1":  3, "2":  4, "3":  5, "4":  6,  "5":  7,  "6": 8,
    "7":  9, "8": 10, "9": 11, "10": 12, "11": 13, "12": 14,
}

print("Configuration loaded.")
print(f"  Train  : {TRAIN_START_YEAR} – {TRAIN_END_YEAR}")
print(f"  Val    : {VAL_START_YEAR} – {VAL_END_YEAR}")
print(f"  Forecast year: {FORECAST_END_YEAR}")


StatementMeta(, d83ac1b2-3777-42b3-b774-bcb8c1822fee, 3, Finished, Available, Finished, False)

Configuration loaded.
  Train  : 2020 – 2025
  Val    : 2026 – 2026
  Forecast year: 2027


---
## Section 2 — Load & Aggregate Enrollment Data

Load `fact_enrollment_annualized_clean`, filter to non-future, non-zero-day rows,
map grades to numeric order, and aggregate to one row per
**school × grade × school year** by counting distinct students.


In [2]:
df_raw = spark.read.table(CLEAN_TABLE)
print(f"Raw rows: {df_raw.count():,}  |  Columns: {len(df_raw.columns)}")

grade_map_expr = F.create_map([F.lit(x) for pair in GRADE_ORDER.items() for x in pair])

df = (
    df_raw
    .filter(F.col("IS_FUTURE_SCHOOL_YEAR") == False)
    .filter(F.col("IS_ZERO_DAY_ENROLLMENT") == False)
    .withColumn("GRADE", F.col("ENTRY_GRADE_LEVEL").cast("string"))
    .withColumn("GRADE_NUMERIC", grade_map_expr[F.col("GRADE")])
    .filter(F.col("GRADE_NUMERIC").isNotNull())
)
print(f"After filtering (no future, no zero-day): {df.count():,}")

df_agg = (
    df
    .groupBy("SCHOOL_KEY", TIME_COL, "GRADE", "GRADE_NUMERIC")
    .agg(F.countDistinct("STUDENT_KEY").alias(LABEL_COL))
)
print(f"Aggregated rows (school × grade × year): {df_agg.count():,}")


StatementMeta(, d83ac1b2-3777-42b3-b774-bcb8c1822fee, 4, Finished, Available, Finished, False)

Raw rows: 9,875,703  |  Columns: 32
After filtering (no future, no zero-day): 2,940,738
Aggregated rows (school × grade × year): 50,912


---
## Section 3 — Lag Features & Feeder Grade

### Same-grade lags
`SAME_GRADE_LAST_YEAR` and `SAME_GRADE_2YR_AGO` are simply `lag(ENROLLMENT, 1)`
and `lag(ENROLLMENT, 2)` within each school-grade partition ordered by year.
These are purely historical — no leakage.

### Feeder grade (`FEEDER_GRADE_LAST_YEAR`, `FEEDER_GRADE_2YR_AGO`)
> *CSR = Grade N this year ÷ Grade N−1 last year*

So the feeder is grade N−1 at the **same school** in the **prior year**.
The join matches `GRADE_NUMERIC == feeder.GRADE_NUMERIC + 1` and
`SCHOOL_YEAR == feeder.SCHOOL_YEAR + 1`.

`HAS_FEEDER_GRADE` is set **before** the coalesce fallback so it accurately
reflects whether a real feeder was found (not the fallback value).


In [3]:
w_sg = Window.partitionBy("SCHOOL_KEY", "GRADE").orderBy(TIME_COL)

# ── 3a. Same-grade lags ──────────────────────────────────────────────────
df_feat = (
    df_agg
    .withColumn("SAME_GRADE_LAST_YEAR", F.lag(LABEL_COL, 1).over(w_sg))
    .withColumn("SAME_GRADE_2YR_AGO",   F.lag(LABEL_COL, 2).over(w_sg))
)

# ── 3b. Feeder grade lookup ──────────────────────────────────────────────
# Domain definition: feeder = grade N-1 at same school, prior year.
feeder_lookup = (
    df_agg.select(
        F.col("SCHOOL_KEY").alias("_fk_school"),
        F.col("GRADE_NUMERIC").alias("_fk_grade_num"),
        F.col(TIME_COL).alias("_fk_year"),
        F.col(LABEL_COL).alias("FEEDER_GRADE_LAST_YEAR"),
    )
)

df_feat = (
    df_feat.join(
        feeder_lookup,
        on=[
            df_feat["SCHOOL_KEY"]    == feeder_lookup["_fk_school"],
            df_feat["GRADE_NUMERIC"] == feeder_lookup["_fk_grade_num"] + 1,
            df_feat[TIME_COL]        == feeder_lookup["_fk_year"] + 1,
        ],
        how="left",
    )
    .drop("_fk_school", "_fk_grade_num", "_fk_year")
    # HAS_FEEDER_GRADE set BEFORE coalesce so it reflects a real feeder, not fallback
    .withColumn(
        "HAS_FEEDER_GRADE",
        F.when(F.col("FEEDER_GRADE_LAST_YEAR").isNotNull(), F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "FEEDER_GRADE_LAST_YEAR",
        F.coalesce(F.col("FEEDER_GRADE_LAST_YEAR"), F.col("SAME_GRADE_LAST_YEAR"))
    )
)

# ── 3c. Feeder grade two years ago (BUG MT-1 fix: build what training references) ──
feeder_2yr_lookup = (
    df_agg.select(
        F.col("SCHOOL_KEY").alias("_f2_school"),
        F.col("GRADE_NUMERIC").alias("_f2_grade_num"),
        F.col(TIME_COL).alias("_f2_year"),
        F.col(LABEL_COL).alias("FEEDER_GRADE_2YR_AGO"),
    )
)

df_feat = (
    df_feat.join(
        feeder_2yr_lookup,
        on=[
            df_feat["SCHOOL_KEY"]    == feeder_2yr_lookup["_f2_school"],
            df_feat["GRADE_NUMERIC"] == feeder_2yr_lookup["_f2_grade_num"] + 1,
            df_feat[TIME_COL]        == feeder_2yr_lookup["_f2_year"] + 2,   # 2 years back
        ],
        how="left",
    )
    .drop("_f2_school", "_f2_grade_num", "_f2_year")
    .withColumn(
        "FEEDER_GRADE_2YR_AGO",
        F.coalesce(F.col("FEEDER_GRADE_2YR_AGO"), F.col("SAME_GRADE_2YR_AGO"))
    )
)

print("Lag + feeder features built.")
print(f"  Rows: {df_feat.count():,}")


StatementMeta(, d83ac1b2-3777-42b3-b774-bcb8c1822fee, 5, Finished, Available, Finished, False)

Lag + feeder features built.
  Rows: 50,912


---
## Section 4 — Cohort Survival Rate (Leak-Free)



### Why it matters
CPS defines CSR as:
> *"CSR = This Year Grade 7 Enrollment ÷ Last Year Grade 6 Enrollment"*

This is a **cross-grade** ratio (feeder → promoted grade). It should **never**
include the current-year label value — that would let the model see the answer.

Both features now contain only information that was available **before** the target year.


In [4]:
# _SR(t) = ENROLLMENT(t) / FEEDER_GRADE_LAST_YEAR(t)  [cross-grade ratio, per domain guide]
# This is the "true" CSR for year t. We compute it but NEVER use it as a feature —
# it contains ENROLLMENT(t) in the numerator and is therefore the label.
# We only use its LAGGED values: SR(t-1), SR(t-2), SR(t-3).

df_feat = df_feat.withColumn(
    "_SR_CURRENT",
    F.when(
        (F.col("FEEDER_GRADE_LAST_YEAR").isNotNull()) & (F.col("FEEDER_GRADE_LAST_YEAR") > 0),
        F.col(LABEL_COL) / F.col("FEEDER_GRADE_LAST_YEAR")
    ).otherwise(F.lit(None))
)

# BUG FE-1 FIX: lag by 1 so COHORT_SURVIVAL_RATE only uses past information
df_feat = df_feat.withColumn(
    "COHORT_SURVIVAL_RATE",
    F.lag("_SR_CURRENT", 1).over(w_sg)
)

# BUG FE-2 FIX: average only over fully lagged SR values (t-1, t-2, t-3)
df_feat = (
    df_feat
    .withColumn("_SR_LAG2", F.lag("_SR_CURRENT", 2).over(w_sg))
    .withColumn("_SR_LAG3", F.lag("_SR_CURRENT", 3).over(w_sg))
    .withColumn(
        "AVG_SURVIVAL_RATE_3YR",
        (
            F.coalesce(F.col("COHORT_SURVIVAL_RATE"), F.lit(0)) +   # lag 1
            F.coalesce(F.col("_SR_LAG2"),             F.lit(0)) +   # lag 2
            F.coalesce(F.col("_SR_LAG3"),             F.lit(0))     # lag 3
        ) / F.greatest(
            F.when(F.col("COHORT_SURVIVAL_RATE").isNotNull(), 1).otherwise(0) +
            F.when(F.col("_SR_LAG2").isNotNull(),             1).otherwise(0) +
            F.when(F.col("_SR_LAG3").isNotNull(),             1).otherwise(0),
            F.lit(1)
        )
    )
    .drop("_SR_CURRENT", "_SR_LAG2", "_SR_LAG3")
)

print("Cohort Survival Rate features built (leak-free).")
print("  COHORT_SURVIVAL_RATE   = lag(ENROLLMENT/FEEDER_GRADE, 1)  [cross-grade, lagged]")
print("  AVG_SURVIVAL_RATE_3YR  = avg of lags 1, 2, 3             [no current-year term]")


StatementMeta(, d83ac1b2-3777-42b3-b774-bcb8c1822fee, 6, Finished, Available, Finished, False)

Cohort Survival Rate features built (leak-free).
  COHORT_SURVIVAL_RATE   = lag(ENROLLMENT/FEEDER_GRADE, 1)  [cross-grade, lagged]
  AVG_SURVIVAL_RATE_3YR  = avg of lags 1, 2, 3             [no current-year term]


---
## Section 5 — School Total & District-Grade Trend Features

### `SCHOOL_TOTAL_LAST_YEAR` (BUG FE-3 fix)

**Original problem:** `SCHOOL_TOTAL_ENROLLMENT` summed current-year enrollment
across all grades. Because one of those grades is the row being predicted, this
aggregation included the label value.

**Fix:** Sum enrollment per school per year, then **lag by 1 year** before joining.
The model now only sees last year's total — never the current year.

### `DISTRICT_GRADE_ENROLLMENT_LAST_YEAR`
This was already correct in the original code (uses a window lag on the district
total). Retained unchanged.

### `IS_MIGRANT_ANOMALY_YEAR`
SY 2022–2024 saw a large surge of migrant students from Venezuela and Central
America, followed by a sharp drop in 2025-26 as federal immigration enforcement
increased. These years are flagged so the model can down-weight them in training.


In [5]:
# ── 5a. School total enrollment — LAGGED (BUG FE-3 fix) ────────────────────
school_totals_raw = (
    df_agg
    .groupBy("SCHOOL_KEY", TIME_COL)
    .agg(F.sum(LABEL_COL).alias("_school_total"))
)
w_school = Window.partitionBy("SCHOOL_KEY").orderBy(TIME_COL)

school_totals = (
    school_totals_raw
    .withColumn(
        "SCHOOL_TOTAL_LAST_YEAR",          # ← lagged by 1 year (no leakage)
        F.lag("_school_total", 1).over(w_school)
    )
    .select("SCHOOL_KEY", TIME_COL, "SCHOOL_TOTAL_LAST_YEAR")
)
# Note: column is now SCHOOL_TOTAL_LAST_YEAR (not SCHOOL_TOTAL_ENROLLMENT)
# Update the FEATURE_COLS list in training accordingly.
df_feat = df_feat.join(school_totals, on=["SCHOOL_KEY", TIME_COL], how="left")

# ── 5b. District-grade enrollment last year (already correct — retained) ─
district_grade_totals = (
    df_agg
    .groupBy("GRADE", TIME_COL)
    .agg(F.sum(LABEL_COL).alias("_dg_total"))
)
w_dg = Window.partitionBy("GRADE").orderBy(TIME_COL)
district_grade_totals = (
    district_grade_totals
    .withColumn(
        "DISTRICT_GRADE_ENROLLMENT_LAST_YEAR",
        F.lag("_dg_total", 1).over(w_dg)
    )
    .drop("_dg_total")
)
df_feat = df_feat.join(district_grade_totals, on=["GRADE", TIME_COL], how="left")

# ── 5c. Migrant anomaly flag ─────────────────────────────────────────────
df_feat = df_feat.withColumn(
    "IS_MIGRANT_ANOMALY_YEAR",
    F.when(F.col(TIME_COL).between(2022, 2024), F.lit(1)).otherwise(F.lit(0))
)

print("School total (lagged), district-grade, and migrant anomaly flag built.")


StatementMeta(, d83ac1b2-3777-42b3-b774-bcb8c1822fee, 7, Finished, Available, Finished, False)

School total (lagged), district-grade, and migrant anomaly flag built.


---
## Section 6 — School-Type Features from `dim_school`

### Deduplication fix
`dim_school` can contain multiple rows per `SCHOOL_KEY` (historical records,
soft-deletes, etc.). Joining without deduplication **fans out** every enrollment
row, silently doubling (or more) the feature table size.

**Fix:** Keep the single most-recently-updated row per `SCHOOL_KEY` using a
row-number window ordered by `LAST_UPDATED_TS` descending before joining.

### School-type features
Per the **domain guide** (Section 4), the four school types have fundamentally
different enrollment dynamics:

| Type | Dynamics |
|------|----------|
| Neighborhood (District) | Address-assigned, predictable from local demographics |
| Selective | Demand-constrained, very stable once enrolled |
| Magnet | Lottery-based, more random year-to-year |
| Charter | Privately managed, can diverge sharply from district trends |


In [6]:
dim = spark.read.table(DIM_TABLE)
print(f"dim_school raw rows       : {dim.count():,}")
print(f"dim_school distinct keys  : {dim.select('SCHOOL_KEY').distinct().count():,}")

# ── BUG FE-4 FIX: deduplicate to 1 row per SCHOOL_KEY ───────────────────
w_dim = Window.partitionBy("SCHOOL_KEY").orderBy(F.col("LAST_UPDATED_TS").desc_nulls_last())
dim_dedup = (
    dim
    .withColumn("_rn", F.row_number().over(w_dim))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)
print(f"dim_school after dedupe   : {dim_dedup.count():,} (1 row per SCHOOL_KEY)")

dim_features = (
    dim_dedup
    .select(
        "SCHOOL_KEY", "GOVERNANCE", "SCHOOL_STATUS", "SCHOOL_SUBTYPE",
        "SCHOOL_GRADES_GROUP", "SELECTIVE_ENROLLMENT_TYPE",
        "ATTENDANCE_BOUNDARY", "ANNUAL_REGIONAL_ANALYSIS_REGION", "COMMUNITY",
    )
    .withColumn(
        "GOVERNANCE_ENCODED",
        F.when(F.upper(F.col("GOVERNANCE")) == "CHARTER",  F.lit(1))
         .when(F.upper(F.col("GOVERNANCE")) == "CONTRACT", F.lit(2))
         .otherwise(F.lit(0))
    )
    .withColumn(
        "IS_SELECTIVE",
        F.when(
            (F.col("SELECTIVE_ENROLLMENT_TYPE").isNotNull()) &
            (F.trim(F.col("SELECTIVE_ENROLLMENT_TYPE")) != "") &
            (F.upper(F.trim(F.col("SELECTIVE_ENROLLMENT_TYPE"))) != "NON"),
            F.lit(1)
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "IS_ATTENDANCE_AREA",
        F.when(F.upper(F.trim(F.col("ATTENDANCE_BOUNDARY"))) == "ATTENDANCE-AREA",
               F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "IS_SMALL_SCHOOL",
        F.when(F.upper(F.trim(F.col("SCHOOL_SUBTYPE"))) == "SMALL",
               F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "IS_HIGH_SCHOOL",
        F.when(F.upper(F.trim(F.col("SCHOOL_GRADES_GROUP"))) == "HIGH",
               F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "IS_SCHOOL_OPEN",
        F.when(F.upper(F.trim(F.col("SCHOOL_STATUS"))) == "OPEN",
               F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "REGION_ENCODED",
        F.when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "South Side",                    F.lit(1))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "West Side",                     F.lit(2))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "North Lakefront",               F.lit(3))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "Northwest Side",                F.lit(4))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "Far Northwest Side",            F.lit(5))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "Bronzeville / South Lakefront", F.lit(6))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "Greater Stony Island",          F.lit(7))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "Greater Calumet",               F.lit(8))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "Greater Midway",                F.lit(9))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "Greater Stockyards",            F.lit(10))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "Pilsen / Little Village",       F.lit(11))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "Near West Side",                F.lit(12))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "Greater Milwaukee Avenue",      F.lit(13))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "Greater Lincoln Park",          F.lit(14))
         .otherwise(F.lit(0))
    )
    .select(
        "SCHOOL_KEY", "GOVERNANCE_ENCODED", "IS_SELECTIVE", "IS_ATTENDANCE_AREA",
        "IS_SMALL_SCHOOL", "IS_HIGH_SCHOOL", "IS_SCHOOL_OPEN", "REGION_ENCODED",
        "GOVERNANCE", "ATTENDANCE_BOUNDARY", "ANNUAL_REGIONAL_ANALYSIS_REGION", "COMMUNITY",
    )
)

df_feat = df_feat.join(dim_features, on="SCHOOL_KEY", how="left")
df_feat = df_feat.fillna({
    "GOVERNANCE_ENCODED": 0, "IS_SELECTIVE": 0, "IS_ATTENDANCE_AREA": 0,
    "IS_SMALL_SCHOOL": 0, "IS_HIGH_SCHOOL": 0, "IS_SCHOOL_OPEN": 1, "REGION_ENCODED": 0,
})

print("\ndim_school features joined.")


StatementMeta(, d83ac1b2-3777-42b3-b774-bcb8c1822fee, 8, Finished, Available, Finished, False)

dim_school raw rows       : 5,830
dim_school distinct keys  : 2,915
dim_school after dedupe   : 2,915 (1 row per SCHOOL_KEY)

dim_school features joined.


---
## Section 7 — Build & Write Final Feature Table

Selects the target grades (1–8, 10–12; K and Grade 9 are entry grades that need
their own models), retains only rows that have at least one year of lag history
(`SAME_GRADE_LAST_YEAR` is not null), and writes to the Lakehouse.

> **Fill-rate check:** Every model feature is printed with its non-null percentage.
> A `⚠` flag appears on any feature below 80% — investigate before training.


In [7]:
final_columns = [
    "SCHOOL_KEY", "GRADE", "GRADE_NUMERIC", TIME_COL,
    LABEL_COL,
    "SAME_GRADE_LAST_YEAR", "SAME_GRADE_2YR_AGO",
    "FEEDER_GRADE_LAST_YEAR", "FEEDER_GRADE_2YR_AGO", "HAS_FEEDER_GRADE",
    "SCHOOL_TOTAL_LAST_YEAR",            # renamed from SCHOOL_TOTAL_ENROLLMENT
    "COHORT_SURVIVAL_RATE",              # leak-free: lag(cross-grade SR, 1)
    "AVG_SURVIVAL_RATE_3YR",             # leak-free: avg(lag 1,2,3)
    "DISTRICT_GRADE_ENROLLMENT_LAST_YEAR",
    "IS_MIGRANT_ANOMALY_YEAR",
    "GOVERNANCE_ENCODED", "IS_SELECTIVE", "IS_ATTENDANCE_AREA",
    "IS_SMALL_SCHOOL", "IS_HIGH_SCHOOL", "REGION_ENCODED",
    "IS_SCHOOL_OPEN",
    "GOVERNANCE", "ATTENDANCE_BOUNDARY", "ANNUAL_REGIONAL_ANALYSIS_REGION", "COMMUNITY",
]

df_final = (
    df_feat
    .filter(F.col("GRADE").isin(TARGET_GRADES))
    .filter(F.col("SAME_GRADE_LAST_YEAR").isNotNull())
    .select(final_columns)
)

final_count = df_final.count()
print(f"Final feature table: {final_count:,} rows × {len(final_columns)} columns")

# Feature fill-rate check
model_features = [
    "SAME_GRADE_LAST_YEAR", "SAME_GRADE_2YR_AGO",
    "FEEDER_GRADE_LAST_YEAR", "FEEDER_GRADE_2YR_AGO", "HAS_FEEDER_GRADE",
    "SCHOOL_TOTAL_LAST_YEAR", "COHORT_SURVIVAL_RATE", "AVG_SURVIVAL_RATE_3YR",
    "DISTRICT_GRADE_ENROLLMENT_LAST_YEAR", "IS_MIGRANT_ANOMALY_YEAR",
    "GOVERNANCE_ENCODED", "IS_SELECTIVE", "IS_ATTENDANCE_AREA",
    "IS_SMALL_SCHOOL", "IS_HIGH_SCHOOL", "REGION_ENCODED",
]
print("\nFeature fill rates:")
for col_name in model_features:
    non_null = df_final.filter(F.col(col_name).isNotNull()).count()
    pct = round(non_null / final_count * 100, 1)
    flag = "✔" if pct >= 80 else "⚠"
    print(f"  {flag}  {col_name:45s}: {pct}%")

df_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(FEATURE_TABLE)
print(f"\n✔ Written: '{FEATURE_TABLE}'")
print(f"  Rows   : {final_count:,}  |  Columns: {len(final_columns)}")


StatementMeta(, d83ac1b2-3777-42b3-b774-bcb8c1822fee, 9, Finished, Available, Finished, False)

Final feature table: 31,750 rows × 26 columns

Feature fill rates:
  ✔  SAME_GRADE_LAST_YEAR                         : 100.0%
  ✔  SAME_GRADE_2YR_AGO                           : 83.8%
  ✔  FEEDER_GRADE_LAST_YEAR                       : 100.0%
  ✔  FEEDER_GRADE_2YR_AGO                         : 84.3%
  ✔  HAS_FEEDER_GRADE                             : 100.0%
  ✔  SCHOOL_TOTAL_LAST_YEAR                       : 100.0%
  ✔  COHORT_SURVIVAL_RATE                         : 84.2%
  ✔  AVG_SURVIVAL_RATE_3YR                        : 100.0%
  ✔  DISTRICT_GRADE_ENROLLMENT_LAST_YEAR          : 100.0%
  ✔  IS_MIGRANT_ANOMALY_YEAR                      : 100.0%
  ✔  GOVERNANCE_ENCODED                           : 100.0%
  ✔  IS_SELECTIVE                                 : 100.0%
  ✔  IS_ATTENDANCE_AREA                           : 100.0%
  ✔  IS_SMALL_SCHOOL                              : 100.0%
  ✔  IS_HIGH_SCHOOL                               : 100.0%
  ✔  REGION_ENCODED                               :

---
## Section 8 — Load Feature Table & Time-Based Split

Reload the saved feature table (or continue with `df_final` in memory), then
split strictly by year.

> **Critical:** **Never split by random sample on a time-series problem.**
> Splitting randomly would let the model train on 2025 data to predict 2022 —
> the future leaks into the past. Always train on years ≤ T−1 and validate on year T.


In [8]:
df = spark.read.table(FEATURE_TABLE)

year_stats = df.select(F.min(TIME_COL).alias("min_y"), F.max(TIME_COL).alias("max_y")).collect()[0]
print(f"Feature table year range: {year_stats['min_y']} – {year_stats['max_y']}")
print(f"Configured  train window: {TRAIN_START_YEAR} – {TRAIN_END_YEAR}")
print(f"Configured    val window: {VAL_START_YEAR}   – {VAL_END_YEAR}")

train_df = df.filter((F.col(TIME_COL) >= TRAIN_START_YEAR) & (F.col(TIME_COL) <= TRAIN_END_YEAR))
val_df   = df.filter((F.col(TIME_COL) >= VAL_START_YEAR)   & (F.col(TIME_COL) <= VAL_END_YEAR))

print(f"\nTrain rows : {train_df.count():,}")
print(f"Val rows   : {val_df.count():,}")


StatementMeta(, d83ac1b2-3777-42b3-b774-bcb8c1822fee, 10, Finished, Available, Finished, False)

Feature table year range: 2020 – 2026
Configured  train window: 2020 – 2025
Configured    val window: 2026   – 2026

Train rows : 26,948
Val rows   : 4,802


---
## Section 9 — NULL Imputation

Medians are computed **from training data only**. Applying training-set medians to
the validation set prevents data leakage from the validation fold into imputation.

Columns filled here are those that are legitimately null for schools in their
first few years of operation (no lag history yet).


In [9]:
FEATURE_COLS = [
    "SAME_GRADE_LAST_YEAR",
    "SAME_GRADE_2YR_AGO",
    "FEEDER_GRADE_LAST_YEAR",
    "FEEDER_GRADE_2YR_AGO",
    "HAS_FEEDER_GRADE",
    "SCHOOL_TOTAL_LAST_YEAR",            # renamed from SCHOOL_TOTAL_ENROLLMENT
    "COHORT_SURVIVAL_RATE",
    "AVG_SURVIVAL_RATE_3YR",
    "DISTRICT_GRADE_ENROLLMENT_LAST_YEAR",
    "IS_MIGRANT_ANOMALY_YEAR",
    "GRADE_NUMERIC",
    "SCHOOL_KEY",
    "GOVERNANCE_ENCODED",
    "IS_SELECTIVE",
    "IS_ATTENDANCE_AREA",
    "IS_SMALL_SCHOOL",
    "IS_HIGH_SCHOOL",
    "REGION_ENCODED",
]

null_fill_cols = [
    "SAME_GRADE_2YR_AGO",
    "FEEDER_GRADE_LAST_YEAR",
    "FEEDER_GRADE_2YR_AGO",
    "COHORT_SURVIVAL_RATE",
    "AVG_SURVIVAL_RATE_3YR",
    "DISTRICT_GRADE_ENROLLMENT_LAST_YEAR",
    "SCHOOL_TOTAL_LAST_YEAR",
]

print("Computing medians from TRAINING data only:")
median_fills = {}
for col_name in null_fill_cols:
    q = train_df.filter(F.col(col_name).isNotNull()).approxQuantile(col_name, [0.5], 0.01)
    median_fills[col_name] = q[0] if q else 0.0
    print(f"  {col_name:45s}: {median_fills[col_name]:.3f}")

train_df = train_df.fillna(median_fills)
val_df   = val_df.fillna(median_fills)
print("\nMedian imputation applied to train and val.")


StatementMeta(, d83ac1b2-3777-42b3-b774-bcb8c1822fee, 11, Finished, Available, Finished, False)

Computing medians from TRAINING data only:
  SAME_GRADE_2YR_AGO                           : 23.000
  FEEDER_GRADE_LAST_YEAR                       : 29.000
  FEEDER_GRADE_2YR_AGO                         : 29.000
  COHORT_SURVIVAL_RATE                         : 0.731
  AVG_SURVIVAL_RATE_3YR                        : 0.627
  DISTRICT_GRADE_ENROLLMENT_LAST_YEAR          : 14185.000
  SCHOOL_TOTAL_LAST_YEAR                       : 458.000

Median imputation applied to train and val.


---
## Section 10 — Sample Weights

School years 2022–2024 contain anomalous enrollment spikes caused by the
Chicago migrant arrival surge (Venezuelan / Central American families).
These years are **not excluded** — the pattern is real — but they are
**down-weighted to 0.3** so they influence training less than normal years.

Validation rows always get weight 1.0 (evaluation should reflect actual conditions).


In [10]:
train_df = train_df.withColumn(
    WEIGHT_COL,
    F.when(F.col("IS_MIGRANT_ANOMALY_YEAR") == 1, F.lit(0.3)).otherwise(F.lit(1.0))
)
val_df = val_df.withColumn(WEIGHT_COL, F.lit(1.0))

migrant_rows = train_df.filter(F.col("IS_MIGRANT_ANOMALY_YEAR") == 1).count()
normal_rows  = train_df.filter(F.col("IS_MIGRANT_ANOMALY_YEAR") == 0).count()
print(f"Sample weights — normal: {normal_rows:,} (w=1.0)  |  anomaly: {migrant_rows:,} (w=0.3)")


StatementMeta(, d83ac1b2-3777-42b3-b774-bcb8c1822fee, 12, Finished, Available, Finished, False)

Sample weights — normal: 12,747 (w=1.0)  |  anomaly: 14,201 (w=0.3)


---
## Section 11 — NaN/Inf Cleaning & Model Training

### Model
GBT Regressor with fixed hyperparameters, operating on a `Pipeline` of:
`StringIndexer(GRADE → GRADE_idx)` → `VectorAssembler` → `GBTRegressor`


In [11]:
def compute_metrics(pred_df, label_col, pred_col):
    """Compute RMSE, MAE, R² using Spark evaluators."""
    evs = {
        "RMSE": RegressionEvaluator(labelCol=label_col, predictionCol=pred_col, metricName="rmse"),
        "MAE":  RegressionEvaluator(labelCol=label_col, predictionCol=pred_col, metricName="mae"),
        "R2":   RegressionEvaluator(labelCol=label_col, predictionCol=pred_col, metricName="r2"),
    }
    return {name: ev.evaluate(pred_df) for name, ev in evs.items()}


# ── BUG MT-2 FIX: type-aware NaN / Inf check ────────────────────────────
float_type_cols = {
    f.name for f in train_df.schema.fields
    if str(f.dataType) in ("DoubleType()", "FloatType()")
}

print("Checking training data for NaN / Inf in feature columns...")
for c in FEATURE_COLS:
    if c in float_type_cols:
        bad_cnt = train_df.filter(
            F.col(c).isNull() | F.isnan(c) |
            (F.col(c) == float("inf")) | (F.col(c) == float("-inf"))
        ).count()
    else:
        bad_cnt = train_df.filter(F.col(c).isNull()).count()
    if bad_cnt > 0:
        print(f"  {c}: {bad_cnt} problematic rows — will be dropped")

def clean_df(sdf, float_cols, all_feature_cols):
    """Drop rows with NaN/Inf (floats) or NULL (ints) in any feature column."""
    out = sdf
    for c in all_feature_cols:
        if c in float_cols:
            out = out.filter(
                ~F.isnan(c) &
                (F.col(c) != float("inf")) &
                (F.col(c) != float("-inf")) &
                F.col(c).isNotNull()
            )
        else:
            out = out.filter(F.col(c).isNotNull())
    return out

train_df_clean = clean_df(train_df, float_type_cols, FEATURE_COLS)
# BUG MT-3 FIX: clean val_df too — was missing in original code
val_df_clean   = clean_df(val_df,   float_type_cols, FEATURE_COLS)

print(f"Train rows before cleaning : {train_df.count():,}")
print(f"Train rows after  cleaning : {train_df_clean.count():,}")
print(f"Val   rows after  cleaning : {val_df_clean.count():,}")

# ── Build pipeline ───────────────────────────────────────────────────────
grade_indexer = StringIndexer(inputCol="GRADE", outputCol="GRADE_idx", handleInvalid="keep")
all_feature_cols = FEATURE_COLS + ["GRADE_idx"]
assembler = VectorAssembler(inputCols=all_feature_cols, outputCol="features", handleInvalid="keep")

gbt = GBTRegressor(
    labelCol=LABEL_COL, featuresCol="features", weightCol=WEIGHT_COL,
    seed=42, maxBins=256,
    maxDepth=BEST_MAX_DEPTH, maxIter=BEST_MAX_ITER,
    stepSize=BEST_STEP_SIZE, subsamplingRate=BEST_SUBSAMPLING_RATE,
)

pipeline = Pipeline(stages=[grade_indexer, assembler, gbt])

print(f"\nTraining GBT (maxDepth={BEST_MAX_DEPTH}, maxIter={BEST_MAX_ITER}, "
      f"stepSize={BEST_STEP_SIZE}, subsampling={BEST_SUBSAMPLING_RATE}) ...")
model = pipeline.fit(train_df_clean)
print("Training complete.")

val_pred = model.transform(val_df_clean)
val_pred = (
    val_pred
    .withColumn("prediction", F.col("prediction").cast(DoubleType()))
    .withColumn(LABEL_COL,    F.col(LABEL_COL).cast(DoubleType()))
)

gbt_metrics = compute_metrics(val_pred, LABEL_COL, "prediction")
best_metrics = gbt_metrics

print(f"\n{'='*60}")
print(f"  GBT MODEL — Validation Metrics (SY {VAL_START_YEAR}–{VAL_END_YEAR})")
print(f"{'='*60}")
print(f"  RMSE : {gbt_metrics['RMSE']:.2f}")
print(f"  MAE  : {gbt_metrics['MAE']:.2f}")
print(f"{'='*60}")


StatementMeta(, d83ac1b2-3777-42b3-b774-bcb8c1822fee, 13, Finished, Available, Finished, False)

Checking training data for NaN / Inf in feature columns...
Train rows before cleaning : 26,948
Train rows after  cleaning : 26,948
Val   rows after  cleaning : 4,802

Training GBT (maxDepth=5, maxIter=100, stepSize=0.1, subsampling=0.8) ...


Training complete.

  GBT MODEL — Validation Metrics (SY 2026–2026)
  RMSE : 35.77
  MAE  : 8.07
  R²   : 0.8740


---
## Section 12 — CSR Baseline Comparison

The correct formula is:
```python
# ✅ Fixed — matches domain definition exactly
csr_prediction = FEEDER_GRADE_LAST_YEAR * AVG_SURVIVAL_RATE_3YR
```
where `AVG_SURVIVAL_RATE_3YR` is the leak-free 3-year average cross-grade ratio
built in Section 4. This is exactly what the current SAS system does.


In [15]:
# CSR baseline = FEEDER_GRADE_LAST_YEAR * AVG_SURVIVAL_RATE_3YR
# Both columns are leak-free (fully lagged) from Section 4 and 5.
# This matches the domain guide definition exactly.
val_pred_csr = (
    val_pred
    .withColumn(
        "csr_prediction",
        (
            F.col("FEEDER_GRADE_LAST_YEAR") *
            F.coalesce(F.col("AVG_SURVIVAL_RATE_3YR"), F.lit(1.0))
        ).cast(DoubleType())
    )
)

csr_metrics = compute_metrics(val_pred_csr, LABEL_COL, "csr_prediction")

print(f"\n{'='*60}")
print(f"  CSR BASELINE  (FEEDER_GRADE_LAST_YEAR × AVG_SURVIVAL_RATE_3YR)")
print(f"  [Domain guide definition: grade N-1 count × historical cross-grade ratio]")
print(f"{'='*60}")
print(f"  RMSE : {csr_metrics['RMSE']:.2f}")
print(f"  MAE  : {csr_metrics['MAE']:.2f}")

rmse_improvement = (csr_metrics['RMSE'] - gbt_metrics['RMSE']) / csr_metrics['RMSE'] * 100
print(f"\n  ML improvement over CSR baseline: {rmse_improvement:.1f}% RMSE reduction")
if rmse_improvement > 0:
    print("  ✔ ML model is BETTER than the CSR baseline.")
else:
    print("  ✘ ML model is NOT better than CSR yet — review features or hyperparameters.")
print(f"{'='*60}")


StatementMeta(, d83ac1b2-3777-42b3-b774-bcb8c1822fee, 17, Finished, Available, Finished, False)


  CSR BASELINE  (FEEDER_GRADE_LAST_YEAR × AVG_SURVIVAL_RATE_3YR)
  [Domain guide definition: grade N-1 count × historical cross-grade ratio]
  RMSE : 4658.94
  MAE  : 249.13

  ML improvement over CSR baseline: 99.2% RMSE reduction
  ✔ ML model is BETTER than the CSR baseline.


---
## Section 13 — Feature Importance

GBT feature importances show how much each feature contributed to split decisions
across all trees. High importance for `COHORT_SURVIVAL_RATE` after the leakage fix
indicates genuine predictive signal — not the model reading the answer off the label.


In [13]:
gbt_model   = model.stages[-1]
importances = gbt_model.featureImportances.toArray()

fi_df = pd.DataFrame({
    "feature":    all_feature_cols,
    "importance": importances,
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("Feature Importance:")
for _, row in fi_df.iterrows():
    bar = "█" * int(row["importance"] * 40)
    print(f"  {row['feature']:45s} {row['importance']:.4f}  {bar}")

v3_features = ["GOVERNANCE_ENCODED", "IS_SELECTIVE", "IS_ATTENDANCE_AREA",
               "IS_SMALL_SCHOOL", "IS_HIGH_SCHOOL", "REGION_ENCODED"]
v3_importance = fi_df[fi_df["feature"].isin(v3_features)]["importance"].sum()
print(f"\n  V3 school-type features combined importance: {v3_importance:.4f}")
print(f"  (> 0.05 indicates school type is meaningfully contributing)")


StatementMeta(, d83ac1b2-3777-42b3-b774-bcb8c1822fee, 15, Finished, Available, Finished, False)

Feature Importance:
  SAME_GRADE_LAST_YEAR                          0.3282  █████████████
  DISTRICT_GRADE_ENROLLMENT_LAST_YEAR           0.1378  █████
  SCHOOL_TOTAL_LAST_YEAR                        0.1377  █████
  GRADE_idx                                     0.1206  ████
  SAME_GRADE_2YR_AGO                            0.0406  █
  AVG_SURVIVAL_RATE_3YR                         0.0371  █
  FEEDER_GRADE_2YR_AGO                          0.0362  █
  COHORT_SURVIVAL_RATE                          0.0361  █
  FEEDER_GRADE_LAST_YEAR                        0.0330  █
  SCHOOL_KEY                                    0.0289  █
  IS_MIGRANT_ANOMALY_YEAR                       0.0128  
  IS_SELECTIVE                                  0.0122  
  REGION_ENCODED                                0.0119  
  GRADE_NUMERIC                                 0.0111  
  HAS_FEEDER_GRADE                              0.0067  
  IS_HIGH_SCHOOL                                0.0063  
  IS_ATTENDANCE_AREA               

---
## Section 14 — Forecast Next School Year

Generates enrollment predictions for `FORECAST_END_YEAR` using the trained model.
Only **open schools** are included — forecasting closed schools is meaningless.

The latest available row per school-grade is used as the feature template.
`SCHOOL_YEAR` is bumped to the forecast year and `IS_MIGRANT_ANOMALY_YEAR` is set
to 0 (the anomaly window is treated as over for future forecasts).


In [14]:
forecast_year = FORECAST_END_YEAR

w_latest = Window.partitionBy("SCHOOL_KEY", "GRADE").orderBy(F.col(TIME_COL).desc())

latest_per_group = (
    df
    .filter(F.col("IS_SCHOOL_OPEN") == 1)
    .fillna(median_fills)
    .withColumn("_rn", F.row_number().over(w_latest))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
    .withColumn(TIME_COL, F.lit(forecast_year))
    .withColumn(WEIGHT_COL, F.lit(1.0))
    .withColumn("IS_MIGRANT_ANOMALY_YEAR", F.lit(0))
)

forecast_pred = model.transform(latest_per_group)

forecast_out = (
    forecast_pred
    .select(
        "SCHOOL_KEY", "GOVERNANCE", "ANNUAL_REGIONAL_ANALYSIS_REGION", "COMMUNITY", "GRADE",
        F.col(TIME_COL).alias("FORECAST_YEAR"),
        F.round("prediction", 0).alias("PREDICTED_ENROLLMENT"),
        "IS_SELECTIVE", "IS_ATTENDANCE_AREA",
    )
    .orderBy("SCHOOL_KEY", "GRADE")
)

print(f"Forecast for SY{forecast_year} (open schools only):")
print(f"  School-grade combinations: {forecast_out.count():,}")
display(forecast_out.limit(30))

FORECAST_TABLE = f"enrollment_forecast_sy{forecast_year}_{FORECAST_TABLE_SUFFIX}"
forecast_out.write.mode("overwrite").saveAsTable(FORECAST_TABLE)
print(f"\n✔ Forecast saved to: '{FORECAST_TABLE}'")


StatementMeta(, d83ac1b2-3777-42b3-b774-bcb8c1822fee, 16, Finished, Available, Finished, False)

Forecast for SY2027 (open schools only):
  School-grade combinations: 5,084


SynapseWidget(Synapse.DataFrame, 4d972ddd-8e89-47cd-ad92-394251695432)


✔ Forecast saved to: 'enrollment_forecast_sy2027_v3_fixed'


---
## Section 15 — Pipeline Summary

| Stage | Table | Description |
|-------|-------|-------------|
| Feature Engineering | `ml_enrollment_feature_table_v3` | 1 row per school × grade × year, all leak-free features |
| Model Training | in-memory `model` | GBT pipeline, fixed hyperparameters |
| Validation Metrics | printed above | RMSE / MAE / R² vs CSR baseline |
| Forecast | `enrollment_forecast_sy{YEAR}_v3_fixed` | Predicted enrollment for open schools only |

### Next Steps
- **Grade 9 separate model** — entry grade; needs GoCPS application data (offers/acceptances from prior year)
- **Kindergarten separate model** — entry grade; needs ILDPH birth data (5-year lag)
- **Hyperparameter tuning** — use the commented-out grid-search cell (Section 6 of the original training notebook) once the leak-free baseline is established
- **Student-level features** — add `%EL`, `%SPED`, `%FRM` per school-grade-year from `dim_student_annual_attrib` as lagged features
